### Simple toy RAGbot

Richard Sear Nov 2025

1st import [chromadb](https://docs.trychroma.com/docs/overview/introduction): a open-source AI application vector database & [ollama](). ollama will need to pull the models ie the model to embed and then the LLM to do the chat, i.e., actually answer the query?

Basic workflow is:

1. Read in pdf that's the source of the text you want to input into the RAGbot, and extract the text from the pdf

2. Split text string from pdf into overlapping chunks. Each chunk is a string of text of length a few hundred characters long (and with some overlap between successive text chunks)

2. Use the *embed* function of an LLM (eg local LLM with ollama) to associate with each text chunk a vector (of length 100+ numbers) that encodes the 'meaning' of the chunk in the sense that LLMs encode meaning or content ...

3. The chunks and associated vectors are then stored in a database (chromadb)

4. The user then asks a question, which itself a string. The 'meaning' vector for this string is generated then the database is searched for the 2 or 3 chunks of text whose 'meaning' vectors are closest to the question text (I think)

5. The question text, the 2 or 3 closest text chunks from the pdf, and some help text that tells the LLM to look for answers in the text chunks are all sent to the *chat* function of an LLM (I use local LLM with ollama). The LLM then generates an answer. With the help text the LLM basically gets: "Based on the following text [retrieved text chunks] Answer the query [question text]". The answer uses the LLM algorithm and so indirectly its training data, as well as the (hopefully) most relevant text from the document, to produce the answer.



In [1]:
import chromadb
from ollama import embed, chat
import pdfplumber #to extract text from pdf
import os

read in a pdf and convert it to text as one long string.... this is for a specific filepath, which will need to be altered ...

In [2]:
def pdf_to_string_range(pdf_path, start_page, end_page):
    """
    Extract text from a range of pages in a PDF.
    Pages are 1-indexed (first page = 1).
    """
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        # Loop through the selected range
        for i in range(start_page - 1, end_page):  # pdfplumber uses 0-based indexing
            page = pdf.pages[i]
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    # Clean up line breaks and spaces
    text = text.replace("\n", " ").replace("\r", " ")
    text = " ".join(text.split())
    return text


def save_text_to_file(text, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(text)

#
current_path=os.getcwd()
print(current_path)
source_filepath=r'C:\Users\rpsea\Downloads\RAGbot1_25Nov'
print(os.listdir(source_filepath))
# Usage
file=source_filepath+r'\FBP_l2.pdf'
pdf_text = pdf_to_string_range(file, start_page=4, end_page=7)
print(pdf_text)

#save_text_to_file(pdf_text, source_filepath+r"\output.txt")
#print("Text extracted and saved to output.txt")

c:\Users\rpsea\Downloads\RAGbot1_25Nov
['FBP_l2.pdf', 'frontiers_assess.txt', 'output.txt', 'RAGbot1.py', 'RAGBot2.ipynb', 'RAGBot2_pdf.ipynb']
Figure2: Schematic (from Wikimedia) illustrating a simple human-made polymer: polyethylene, alsocalledpolythene. Polymersaremoleculesthataremadefromlonglinearchainsofmolecules — called monomers — strung together one after another. For polyethylene, every monomer is the same — so polyethylene is a homopolymer. The monomer is just a single carbon atom (shown as C) plus two chemically attached hydrogen atoms (shown as H). In the () are two monomers, and the n indicates that the polymer repeats this n ≫ 1 times. All polyethylene is is verylongmoleculesmadebystringingthousandsofthesemonomersofcarbonandhydrogenend to end. Proteins, DNA and RNA are also polymers but these are heteropolymers, which means, unlike homopolymers, that the monomers are not all the same. Cellulose is a homopolymer, where the monomers are glucose. We are (mostly) made of Wate

chop the text up into chunks and use a model within ollama to generate embeddings, then add chunks and embedding to chromadb collection

In [3]:

'''
file=source_filepath+r'\output.txt'
# Step 1: Read in a text file
with open(file, "r", encoding="utf-8") as f:
    text = f.read()
'''

chunk_length=500
# Step 2: Chunk the text
def chunk_text(text, chunk_size=chunk_length, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(pdf_text)


# Step 3: Setup ChromaDB
client = chromadb.Client()
collection = client.create_collection(name="my_collection")


# Step 4: Batch embeddings with Ollama
# Pass the whole list of chunks at once
#embedding_model="nomic-embed-text"
embedding_model='all-miniLM'
embeddings = embed(model=embedding_model, input=chunks)["embeddings"]

print('embeddings generated with model ',embedding_model,' \n')
print('1st chunk     ',chunks[0])
print('chunks have length ',len(chunks[0]))
print('has embedding ',embeddings[0])
print('embedding vector is of length ',len(embeddings[0]))


# Add all chunks + embeddings to ChromaDB in one call
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
print('chunks and embeddings added to chromadb collection \n')


<>:2: SyntaxWarning: invalid escape sequence '\o'
<>:2: SyntaxWarning: invalid escape sequence '\o'
C:\Users\rpsea\AppData\Local\Temp\ipykernel_6024\2884923889.py:2: SyntaxWarning: invalid escape sequence '\o'
  file=source_filepath+r'\output.txt'


embeddings generated with model  all-miniLM  

1st chunk      Figure2: Schematic (from Wikimedia) illustrating a simple human-made polymer: polyethylene, alsocalledpolythene. Polymersaremoleculesthataremadefromlonglinearchainsofmolecules — called monomers — strung together one after another. For polyethylene, every monomer is the same — so polyethylene is a homopolymer. The monomer is just a single carbon atom (shown as C) plus two chemically attached hydrogen atoms (shown as H). In the () are two monomers, and the n indicates that the polymer repeats this
chunks have length  500
has embedding  [-0.030655803, -0.013029278, 0.029669879, 0.023959557, -0.056357764, -0.010047781, 0.019236362, 0.044487007, -0.00073686504, -0.0043317075, 0.035595503, 0.006030844, -0.02210773, 0.042948607, -0.032716453, 0.020805446, -0.15241057, -0.011443731, -0.010921749, -0.008252472, 0.07222678, -0.01215243, -0.042915158, -0.0010506297, -0.014787247, 0.028964495, -0.0049436963, 0.002392702, 0.074070096, -0

now ask a question from the terminal,  use ollama again to generate an emebeding for the question text, and then pass this to the chromadb collection of the embedded source text, retrive a small number of text chunks from the source test hopefully related to the text query. Then pass both the question and the retrieved chunks to a LLM using a model in ollama chat and ask the LLM to answer the question based on the retrieved chunks.

In [4]:

# Step 5: Query ChromaDB
#query_text = "Summarize the main idea of the document"
#query_text = "How many Papers are there?"
#query_text = "How many topics are there?"
query_text=input('ask anything! :  ')
query_emb = embed(model=embedding_model, input=[query_text])["embeddings"][0]
print(query_emb)

n_chunks_passed=3
print('retrieving and passing ',n_chunks_passed,' text chunks each of length ',chunk_length)
results = collection.query(
    query_embeddings=[query_emb],
    n_results=n_chunks_passed
)

retrieved_chunks = results["documents"][0]

print('chunks of text retrieved from the source and passed to LLM are \n',retrieved_chunks)

'''
# Step 6: Use Gemma LLM for reasoning
response = chat(model="gemma:2b", messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Based on the following text:\n\n{retrieved_chunks}\n\nAnswer the query: {query_text}"}
])

print(response["message"]["content"])
'''

# --- Streaming response from Gemma ---
llm_model="gemma:2b"
#llm_model='qwen3-vl:4b'
print(' using LLM ',llm_model)
stream = chat(
    model=llm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"Based on the following text:\n\n{retrieved_chunks}\n\nAnswer the query: {query_text}"}
    ],
    stream=True  # enable streaming
)

print(llm_model," says:\n")
for chunk in stream:
    # Each chunk contains partial output
    if "message" in chunk and "content" in chunk["message"]:
        print(chunk["message"]["content"], end="", flush=True)

print("\n\n--- End of streamed response ---")

[-0.04467454, -0.0072343615, -0.02843518, -0.0075524542, -0.050055068, -0.01066713, 0.0032593962, 0.042462375, -0.020882769, 0.031142289, 0.03487887, 0.09174361, -0.09281541, 0.043513242, -0.049047083, 0.024034651, -0.0872642, 0.049368337, 0.060957946, 0.0189676, -0.014880031, 0.007275904, 0.035541162, -0.010097925, -0.028742036, 0.057493735, 0.012666977, -0.037745453, 0.030433359, -0.06158282, 0.027753696, 0.04749961, -0.0122096045, -0.008880161, 0.053083256, 0.034972943, 0.030662624, 0.044041835, 0.014028015, 0.014233305, 0.06570222, -0.03553334, -0.009588672, 0.04109553, -0.043666136, 0.013897042, -0.018162612, -0.002342819, -0.03449922, -0.008132585, -0.018272089, -0.00033072813, -0.011545645, -0.08051201, -0.04577729, 0.10221371, 0.00021835517, -0.06178937, 0.0018802414, 0.003902954, 0.014184407, 0.004027119, -0.015950035, 0.0711223, 0.11705903, 0.0441401, 0.1034798, -0.017966142, 0.025225185, -1.1485369e-05, 0.015998157, -0.010329254, 0.043745276, 0.03904296, 0.037181, -0.0704862